# Research Pipeline - Multi-Agent Workflow

This notebook implements a modular research → analysis → report workflow using the OpenAI Agents SDK.

## Architecture
- **Researcher**: Searches the web using Tavily API and gathers sources
- **Analyst**: Analyzes trends, risks, and insights from research
- **Writer**: Generates structured output (executive summary, report, follow-up questions)

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q openai-agents python-dotenv pydantic requests

In [ ]:
import os
import json
import requests
import uuid
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

# Import OpenAI Agents SDK
from agents import Agent, Runner, function_tool, SQLiteSession

# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

if not OPENAI_API_KEY or not TAVILY_API_KEY:
    raise ValueError("Missing API keys. Please set OPENAI_API_KEY and TAVILY_API_KEY in .env file")

print("✓ Environment loaded successfully")

## 2. Tavily Search Tool

In [ ]:
@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily API.
    
    Args:
        query: The search query
        max_results: Maximum number of results to return (default: 5)
    
    Returns:
        JSON string containing search results with titles, URLs, and content
    """
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results
    }
    
    try:
        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Format results
        results = []
        for result in data.get("results", []):
            results.append({
                "title": result.get("title", ""),
                "url": result.get("url", ""),
                "content": result.get("content", "")
            })
        
        return json.dumps(results, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})

print("✓ Tavily search tool created")

## 3. Output Models (Pydantic)

In [ ]:
class ResearchSummary(BaseModel):
    """Output from Researcher agent"""
    query: str
    summary: str
    sources: List[str]

class AnalysisInsights(BaseModel):
    """Output from Analyst agent"""
    trends: List[str]
    risks: List[str]
    key_insights: List[str]

class FinalReport(BaseModel):
    """Output from Writer agent"""
    executive_summary: str
    markdown_report: str
    follow_up_questions: List[str]

print("✓ Output models defined")

## 4. Agent Definitions

In [ ]:
def create_researcher_agent():
    """Create the Researcher agent with Tavily search capability."""
    return Agent(
        name="Researcher",
        model="gpt-4o-mini",  # Using cheaper model to conserve credits
        instructions="""
You are a thorough research assistant.

Your job is to:
1. Use the tavily_search tool to gather information from the web
2. Search multiple times with different queries if needed to get comprehensive information
3. Compile a detailed summary of your findings
4. List all sources you used

IMPORTANT: Always use the tavily_search tool before providing your summary.
Do not make up information - only use what you find from searches.
""",
        tools=[tavily_search],
        output_schema=ResearchSummary
    )

def create_analyst_agent():
    """Create the Analyst agent."""
    return Agent(
        name="Analyst",
        model="gpt-4o-mini",
        instructions="""
You are an analytical expert who reviews research findings.

Your job is to:
1. Review the research summary provided
2. Identify emerging trends
3. Highlight potential risks or concerns
4. Extract key insights that are actionable

Be concise but thorough. Focus on what matters most.
""",
        output_schema=AnalysisInsights
    )

def create_writer_agent():
    """Create the Writer agent."""
    return Agent(
        name="Writer",
        model="gpt-4o-mini",
        instructions="""
You are a professional report writer.

Your job is to:
1. Create a concise executive summary (2-3 paragraphs)
2. Write a well-formatted markdown report with:
   - Introduction
   - Key Findings (from research)
   - Analysis (trends, risks, insights)
   - Conclusion
3. Generate 3-5 follow-up questions for deeper investigation

Use clear headings, bullet points, and proper markdown formatting.
Make the report professional and easy to read.
""",
        output_schema=FinalReport
    )

print("✓ Agent factory functions created")

## 5. Pipeline Orchestration

In [ ]:
def run_pipeline(user_query: str, researcher_agent: Agent, session_id: str = None):
    """
    Orchestrate the multi-agent research pipeline.
    
    Args:
        user_query: The research question/topic
        researcher_agent: The researcher agent (modular, can be swapped)
        session_id: Optional session ID for SQLite persistence
    
    Returns:
        FinalReport object with executive summary, markdown report, and follow-up questions
    """
    # Generate unique session ID if not provided
    if session_id is None:
        session_id = str(uuid.uuid4())
    
    print(f"\n{'='*60}")
    print(f"Starting Research Pipeline")
    print(f"Session ID: {session_id}")
    print(f"Query: {user_query}")
    print(f"{'='*60}\n")
    
    # Create SQLite session for context sharing
    session = SQLiteSession(session_id=session_id)
    
    # Stage 1: Research
    print("[1/3] 🔍 Researching...")
    runner = Runner(researcher_agent, session=session)
    research_result = runner.run(user_query)
    research_summary = research_result.output
    print(f"✓ Research completed: {len(research_summary.sources)} sources found")
    
    # Stage 2: Analysis
    print("\n[2/3] 📊 Analyzing...")
    analyst = create_analyst_agent()
    runner = Runner(analyst, session=session)
    analysis_prompt = f"""
Analyze this research summary:

Query: {research_summary.query}
Summary: {research_summary.summary}
Sources: {', '.join(research_summary.sources)}
"""
    analysis_result = runner.run(analysis_prompt)
    analysis_insights = analysis_result.output
    print(f"✓ Analysis completed: {len(analysis_insights.trends)} trends, {len(analysis_insights.risks)} risks identified")
    
    # Stage 3: Report Writing
    print("\n[3/3] ✍️  Writing report...")
    writer = create_writer_agent()
    runner = Runner(writer, session=session)
    writer_prompt = f"""
Create a comprehensive report based on:

RESEARCH:
- Query: {research_summary.query}
- Summary: {research_summary.summary}
- Sources: {', '.join(research_summary.sources)}

ANALYSIS:
- Trends: {', '.join(analysis_insights.trends)}
- Risks: {', '.join(analysis_insights.risks)}
- Key Insights: {', '.join(analysis_insights.key_insights)}
"""
    final_result = runner.run(writer_prompt)
    final_report = final_result.output
    print("✓ Report completed")
    
    print(f"\n{'='*60}")
    print("Pipeline Complete!")
    print(f"{'='*60}\n")
    
    return final_report

print("✓ Pipeline function defined")

## 6. Example Usage

In [ ]:
# Create researcher agent (modular - can be swapped out later)
researcher = create_researcher_agent()

# Example query
query = "What are the latest developments in AI agent frameworks in 2025?"

# Run the pipeline
report = run_pipeline(query, researcher)

## 7. Display Results

In [ ]:
from IPython.display import display, Markdown

print("\n" + "="*60)
print("EXECUTIVE SUMMARY")
print("="*60 + "\n")
print(report.executive_summary)

print("\n" + "="*60)
print("FULL REPORT")
print("="*60 + "\n")
display(Markdown(report.markdown_report))

print("\n" + "="*60)
print("FOLLOW-UP QUESTIONS")
print("="*60 + "\n")
for i, question in enumerate(report.follow_up_questions, 1):
    print(f"{i}. {question}")

## 8. Test with Different Queries

In [ ]:
# Test with different queries
test_queries = [
    "What are the security implications of LLM-based autonomous agents?",
    "How is quantum computing affecting cybersecurity in 2025?"
]

# Uncomment to run additional tests
# for test_query in test_queries:
#     report = run_pipeline(test_query, researcher)
#     display(Markdown(f"### Query: {test_query}\n\n{report.markdown_report}"))